# Chapter 5 Exercise Solutions

Worked answers for the three exercises in [`ch05_patient_journey.md`](ch05_patient_journey.md). The judgment call at the end of each answer matters more than the pandas.

## Setup

Rebuild the chapter cohort and fills once.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, "ch05_journey/scripts")

import pandas as pd

from episode_construction import build_newly_observed_cohort, load_chapter3_data
from lot import construct_lines_of_therapy, lot_entry_shares
from survival import km_curve, km_estimate_at, km_median

tables = load_chapter3_data(Path("ch03_data/output_data/generated_data"))
cohort, _ = build_newly_observed_cohort(tables, minimum_lookback_days=180, minimum_followup_days=90)
basket = tables["products"]["product_name"].tolist()
pharmacy = tables["pharmacy_claims"]
paid_all = pharmacy[
    pharmacy.transaction_type.eq("PAID")
    & pharmacy.product_name.isin(basket)
    & pharmacy.patient_id.isin(cohort.patient_id)
].copy()
lines_base, initiators = construct_lines_of_therapy(paid_all, cohort)
print(f"cohort {len(cohort):,} | new to therapy {initiators.new_to_therapy.sum():,} | lines {len(lines_base):,}")

cohort 5,637 | new to therapy 3,415 | lines 3,443


## Exercise 1: Flip the restart rule

Rerun the construction with `restart_advances_line=False`, so a regimen product returning after the allowable gap resumes its old line instead of opening a new one.

In [2]:
lines_flip, _ = construct_lines_of_therapy(paid_all, cohort, restart_advances_line=False)

for label, lines in [("restart advances (base)", lines_base), ("restart resumes", lines_flip)]:
    l1 = lines[lines.line_number.eq(1)]
    entries = lot_entry_shares(lines)
    line1_entries = int(entries.loc[entries.position.eq("Line 1"), "line_entries"].sum())
    print(f"{label}: {len(lines):,} lines | L1 discontinued {l1.end_reason.eq('Discontinued').mean():.1%}"
          f" | Roventra L1 entries {line1_entries:,}")

restart advances (base): 3,443 lines | L1 discontinued 42.8% | Roventra L1 entries 2,798
restart resumes: 3,443 lines | L1 discontinued 42.8% | Roventra L1 entries 2,798


**Why nothing moves.** The generator builds tight refill chains: 99% of same-product refills arrive within 11 days of supply ending, so the 60-day gap almost never closes a line that the same product later reopens. The restart rule has nothing to act on.

**Methods note.** "Restart handling (new line versus resumed line) was tested and does not affect any reported quantity in this data; with longer refill gaps, holiday supply interruptions, or 90-day mail-order fills, the choice can move both the line count and the discontinuation rate, and should be re-tested per source."

## Exercise 2: Payer journeys

Initial-regimen persistence by payer. Switch, addition, restart, and confirmed discontinuation are events; only administrative censoring is censored.

In [3]:
payer = cohort.set_index("patient_id")["payer_id"]
l1 = lines_base[lines_base.line_number.eq(1)].copy()
l1["payer_id"] = l1.patient_id.map(payer)

rows = []
for payer_id, grp in l1.groupby("payer_id"):
    curve = km_curve(grp.line_days, ~grp.end_reason.eq("Censored"))
    rows.append({
        "payer_id": payer_id,
        "patients": len(grp),
        "median_days": km_median(curve),
        "still_on_at_180": round(km_estimate_at(curve, 180), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))

payer_id  patients  median_days  still_on_at_180
  PAY001       396        120.0            0.184
  PAY002       438        122.0            0.296
  PAY003       420        119.0            0.200
  PAY004       460        113.0            0.186
  PAY005       426        102.0            0.121
  PAY006       416         94.0            0.169
  PAY007       425         98.0            0.187
  PAY008       434        119.0            0.190


**Reading the null.** The payer curves sit within a few days of one another, which is how the generator built them. On real data a payer gap this small would still not be reportable on its own: payer is confounded with formulary position, age mix, and channel. Before believing a non-null, stratify by the access status from Chapter 3's formulary table and check whether the gap survives; Chapter 6 takes up access directly, and Chapters 10 and 11 supply the designs that turn a stratified difference into evidence.

## Exercise 3: The immature tail

Move `study_end` to 2025-01-31, the raw edge of the data, and measure what the extra month of structurally incomplete claims does.

In [4]:
cohort_raw, _ = build_newly_observed_cohort(
    tables, minimum_lookback_days=180, minimum_followup_days=90,
    study_end=pd.Timestamp("2025-01-31"),
)
paid_raw = pharmacy[
    pharmacy.transaction_type.eq("PAID")
    & pharmacy.product_name.isin(basket)
    & pharmacy.patient_id.isin(cohort_raw.patient_id)
].copy()
lines_raw, _ = construct_lines_of_therapy(paid_raw, cohort_raw)

for label, cohort_x, lines_x in [("mature (2024-12-31)", cohort, lines_base),
                                 ("raw edge (2025-01-31)", cohort_raw, lines_raw)]:
    l1 = lines_x[lines_x.line_number.eq(1)]
    print(f"{label}: cohort {len(cohort_x):,} | lines {len(lines_x):,}"
          f" | L1 discontinued {l1.end_reason.eq('Discontinued').mean():.1%}"
          f" | L1 censored {l1.end_reason.eq('Censored').mean():.1%}")

# Where does the difference concentrate?
jan_end = cohort_raw.followup_end.gt(pd.Timestamp("2024-12-31"))
l1_raw = lines_raw[lines_raw.line_number.eq(1)].copy()
l1_raw["tail"] = l1_raw.patient_id.map(cohort_raw.set_index("patient_id").followup_end).gt(pd.Timestamp("2024-12-31"))
print("\nL1 discontinued share by follow-up tail:")
print(l1_raw.groupby("tail").end_reason.apply(lambda s: round(s.eq("Discontinued").mean(), 3)).to_string())

mature (2024-12-31): cohort 5,637 | lines 3,443 | L1 discontinued 42.8% | L1 censored 56.3%
raw edge (2025-01-31): cohort 5,932 | lines 3,609 | L1 discontinued 52.0% | L1 censored 47.1%

L1 discontinued share by follow-up tail:
tail
True    0.52


**Reading the artifact.** Extending the study end pulls in index dates and fills whose claims have not finished arriving (Chapter 3, section 3.7.1). The discontinuation rule fires when supply plus 60 days fits inside follow-up, so an extra month of incomplete data converts censored lines into apparent stops that a mature refresh will quietly undo. The operational rule: fix the study end at a mature boundary, state it in the estimand, and never compare journey KPIs computed at different maturities.